# N4 — Regímenes de metáforas (Visualización)
## AI-MELT: Nivel 4 — Exploración visual

**Prerequisito:** Haber ejecutado `N4_regimenes.ipynb` (procesamiento).

---

### Archivos de entrada esperados (en `./outputs/N4/`)

| Archivo | Columnas clave | Descripción |
|---|---|---|
| `n4_regimenes.csv` | ID_regimen, nombre_regimen, n_escenarios, coherencia_interna, modularity_score, metodo_clustering | Un régimen por fila |
| `n4_escenario_regimen.csv` | ID_regimen, ID_escenario | Tabla M:N escenarios ↔ regímenes |
| `n4_ejes_valorativos.csv` | ID_eje, ID_regimen, polo_a, polo_b, varianza_explicada | Dimensiones bipolares |
| `n4_metaforas_derivadas.csv` | ID_derivada, ID_regimen, metafora_derivada, tipo_inferencia, escenarios_origen, validada_contra_compendio | Transitividad |
| `n4_grafo_cache.json` | nodes (id, label, estatus, degree_centrality, betweenness), edges (source, target, weight) | Grafo serializado |
| `n4_embeddings_cache.npz` | scenario_embeddings, pca_coords, sim_matrix | Embeddings y PCA |
| `n4_artifacts.json` | scenario_ids, labels_*, best_method, modularity, pca_variance_explained | Labels y métricas |

### Archivos de N3 y N2 también usados

| Archivo | Uso |
|---|---|
| `n3_escenarios.csv` | Nombres y estatus de escenarios |
| `n2_metaforas_convencionales.csv` | Para Sankey multinivel |

---

### Visualizaciones generadas
1. Grafo del campo metafórico (nodos por régimen, tamaño por frecuencia)
2. UMAP 2D de escenarios coloreado por régimen
3. Barras de coherencia interna por régimen
4. Tabla de metáforas derivadas
5. Sankey multinivel: metáforas convencionales → escenarios → regímenes

## 1. Configuración y carga

In [ ]:
# ============================================================
# 1. IMPORTS Y CARGA
# ============================================================
import json
import os
import warnings

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

N2_DIR = "./outputs/N2/"
N3_DIR = "./outputs/N3/"
N4_DIR = "./outputs/N4/"
VIZ_DIR = "./outputs/N4/viz/"
os.makedirs(VIZ_DIR, exist_ok=True)

df_reg = pd.read_csv(os.path.join(N4_DIR, "n4_regimenes.csv"))
df_esc_reg = pd.read_csv(os.path.join(N4_DIR, "n4_escenario_regimen.csv"))
df_ejes = pd.read_csv(os.path.join(N4_DIR, "n4_ejes_valorativos.csv"))
df_deriv = pd.read_csv(os.path.join(N4_DIR, "n4_metaforas_derivadas.csv"))
df_esc = pd.read_csv(os.path.join(N3_DIR, "n3_escenarios.csv"))
df_conv = pd.read_csv(os.path.join(N2_DIR, "n2_metaforas_convencionales.csv"))

with open(os.path.join(N4_DIR, "n4_grafo_cache.json")) as f:
    grafo_data = json.load(f)
cache = np.load(os.path.join(N4_DIR, "n4_embeddings_cache.npz"))
pca_coords = cache["pca_coords"]
with open(os.path.join(N4_DIR, "n4_artifacts.json")) as f:
    artifacts = json.load(f)

scenario_ids = artifacts["scenario_ids"]
best_method = artifacts["best_method"]
selected_labels = np.array(artifacts[f"labels_{'louvain' if best_method == 'Louvain' else 'dbscan'}"])

print("✓ Datos cargados:")
print(f"  Regímenes: {len(df_reg)}")
print(f"  Escenarios en regímenes: {len(df_esc_reg)}")
print(f"  Ejes valorativos: {len(df_ejes)}")
print(f"  Metáforas derivadas: {len(df_deriv)}")

## 2. Grafo del campo metafórico

In [ ]:
# ============================================================
# 2. GRAFO DEL CAMPO METAFÓRICO
# ============================================================

# Reconstruir grafo desde cache
G = nx.Graph()
for node in grafo_data["nodes"]:
    G.add_node(node["id"], **{k: v for k, v in node.items() if k != "id"})
for edge in grafo_data["edges"]:
    G.add_edge(edge["source"], edge["target"], weight=edge["weight"])

# Colores por régimen
n_regimes = len(df_reg)
regime_colors = plt.cm.get_cmap("tab10", max(n_regimes, 1))
regime_map = dict(zip(df_esc_reg["ID_escenario"], df_esc_reg["ID_regimen"], strict=False))
regime_id_to_idx = {r: i for i, r in enumerate(df_reg["ID_regimen"])}

fig, ax = plt.subplots(figsize=(16, 12))
pos_layout = nx.spring_layout(G, seed=42, k=2, weight="weight")

# Dibujar aristas
nx.draw_networkx_edges(G, pos_layout, alpha=0.15, width=0.5, ax=ax)

# Dibujar nodos coloreados por régimen, tamaño por centralidad
for node in G.nodes():
    reg_id = regime_map.get(node, "")
    reg_idx = regime_id_to_idx.get(reg_id, -1)
    color = regime_colors(reg_idx) if reg_idx >= 0 else "lightgray"
    size = 100 + G.nodes[node].get("degree_centrality", 0) * 2000
    ax.scatter(*pos_layout[node], c=[color], s=size, alpha=0.8,
               edgecolors='white', linewidth=0.5, zorder=3)

# Labels
labels = {n: G.nodes[n].get("label", n)[:20] for n in G.nodes()}
nx.draw_networkx_labels(G, pos_layout, labels, font_size=6, ax=ax)

# Leyenda
from matplotlib.patches import Patch

legend_patches = [Patch(facecolor=regime_colors(i), label=f"{row['ID_regimen']}: {row['nombre_regimen'][:35]}")
                   for i, (_, row) in enumerate(df_reg.iterrows())]
ax.legend(handles=legend_patches, loc='upper left', fontsize=7, title="Regímenes")
ax.set_title(f"Campo metafórico ({G.number_of_nodes()} escenarios, {G.number_of_edges()} aristas, {n_regimes} regímenes)")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_grafo_campo_metaforico.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 3. UMAP 2D de escenarios coloreado por régimen

In [ ]:
# ============================================================
# 3. UMAP 2D
# ============================================================
try:
    import umap
    scenario_embs = cache["scenario_embeddings"]
    reducer = umap.UMAP(n_components=2, random_state=42, metric="cosine",
                          n_neighbors=min(15, len(scenario_embs)-1))
    coords = reducer.fit_transform(scenario_embs)

    fig, ax = plt.subplots(figsize=(14, 10))
    for label in sorted(set(selected_labels)):
        mask = selected_labels == label
        color = "lightgray" if label == -1 else regime_colors(label % 10)
        lbl = "Outliers" if label == -1 else f"Régimen {label}"
        ax.scatter(coords[mask, 0], coords[mask, 1], c=[color], label=lbl, s=80, alpha=0.7, edgecolors='white')
        for j in np.where(mask)[0]:
            node_id = scenario_ids[j]
            name = G.nodes[node_id].get("label", "") if node_id in G.nodes else ""
            ax.annotate(name[:18], (coords[j, 0], coords[j, 1]), fontsize=6, alpha=0.7)

    ax.set_title(f"UMAP 2D — escenarios por régimen ({best_method})")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_umap_regimenes.png"), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
except ImportError:
    print("⚠ umap-learn no instalado.")

## 4. Coherencia interna por régimen

In [ ]:
# ============================================================
# 4. BARRAS DE COHERENCIA INTERNA
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coherencia
axes[0].barh(df_reg["ID_regimen"], df_reg["coherencia_interna"],
              color=[regime_colors(i) for i in range(len(df_reg))])
axes[0].set_xlabel("Coherencia interna (similitud coseno promedio)")
axes[0].set_title("Coherencia interna por régimen")

# Número de escenarios
axes[1].barh(df_reg["ID_regimen"], df_reg["n_escenarios"],
              color=[regime_colors(i) for i in range(len(df_reg))])
axes[1].set_xlabel("Número de escenarios")
axes[1].set_title("Tamaño de cada régimen")

plt.tight_layout()
plt.savefig(os.path.join(VIZ_DIR, "viz_coherencia_regimenes.png"), dpi=150, bbox_inches='tight')
plt.show()
plt.close()

## 5. Metáforas derivadas por transitividad

In [ ]:
# ============================================================
# 5. TABLA DE METÁFORAS DERIVADAS (Plotly)
# ============================================================

if len(df_deriv) > 0:
    cols = ["metafora_derivada", "tipo_inferencia", "ID_regimen", "razonamiento", "validada_contra_compendio"]
    cols_exist = [c for c in cols if c in df_deriv.columns]
    df_display = df_deriv[cols_exist].copy()
    df_display.columns = [c.replace("_", " ").title() for c in cols_exist]

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(df_display.columns), fill_color='#1F4E79',
                     font=dict(color='white', size=10), align='left'),
        cells=dict(values=[df_display[c] for c in df_display.columns],
                    font=dict(size=9), align='left', height=25)
    )])
    fig.update_layout(title=f"Metáforas derivadas por transitividad ({len(df_deriv)})",
                       height=max(400, 30*len(df_deriv)))
    out_path = os.path.join(VIZ_DIR, "viz_metaforas_derivadas.html")
    pio.write_html(fig, out_path, include_plotlyjs="cdn")
    print(f"✓ {out_path}")
    fig.show()
else:
    print("⚠ No hay metáforas derivadas.")

## 6. Sankey multinivel: metáforas convencionales → escenarios → regímenes

In [ ]:
# ============================================================
# 6. SANKEY MULTINIVEL (Plotly)
# ============================================================

# Obtener conexiones: convencional → escenario (vía n3_escenarios.csv)
# y escenario → régimen (vía n4_escenario_regimen.csv)

# Nivel 1: metáforas convencionales → escenarios
conv_to_esc = df_esc[["ID_metaconvencional_base", "ID_escenario", "nombre_escenario"]].dropna()
conv_to_esc = conv_to_esc.merge(
    df_conv[["ID_metaconvencional", "metafora_conceptual"]].drop_duplicates(),
    left_on="ID_metaconvencional_base", right_on="ID_metaconvencional", how="left"
)

# Nivel 2: escenarios → regímenes
esc_to_reg = df_esc_reg.merge(df_reg[["ID_regimen", "nombre_regimen"]], on="ID_regimen", how="left")

# Construir nodos
mc_labels = sorted(conv_to_esc["metafora_conceptual"].dropna().unique().tolist())
esc_labels = sorted(conv_to_esc["nombre_escenario"].dropna().unique().tolist())
reg_labels = sorted(df_reg["nombre_regimen"].dropna().unique().tolist())

all_labels = ([f"[MC] {m[:40]}" for m in mc_labels] +
              [f"[ESC] {e[:35]}" for e in esc_labels] +
              [f"[REG] {r[:35]}" for r in reg_labels])
node_idx = {lbl: i for i, lbl in enumerate(all_labels)}

sources, targets, values, colors_link = [], [], [], []
regime_color_map = {row["nombre_regimen"]: regime_colors(i) for i, (_, row) in enumerate(df_reg.iterrows())}

# MC → ESC
for _, row in conv_to_esc.iterrows():
    mc_lbl = f"[MC] {str(row.get('metafora_conceptual',''))[:40]}"
    esc_lbl = f"[ESC] {str(row.get('nombre_escenario',''))[:35]}"
    if mc_lbl in node_idx and esc_lbl in node_idx:
        sources.append(node_idx[mc_lbl])
        targets.append(node_idx[esc_lbl])
        values.append(1)
        colors_link.append("rgba(150,150,150,0.3)")

# ESC → REG
for _, row in esc_to_reg.iterrows():
    esc_name = df_esc[df_esc["ID_escenario"] == row["ID_escenario"]]["nombre_escenario"]
    if len(esc_name) > 0:
        esc_lbl = f"[ESC] {str(esc_name.iloc[0])[:35]}"
        reg_lbl = f"[REG] {str(row.get('nombre_regimen',''))[:35]}"
        if esc_lbl in node_idx and reg_lbl in node_idx:
            sources.append(node_idx[esc_lbl])
            targets.append(node_idx[reg_lbl])
            values.append(1)
            c = regime_color_map.get(row.get("nombre_regimen", ""), (0.5, 0.5, 0.5, 1))
            if isinstance(c, tuple):
                colors_link.append(f"rgba({int(c[0]*255)},{int(c[1]*255)},{int(c[2]*255)},0.5)")
            else:
                colors_link.append("rgba(100,100,200,0.5)")

if sources:
    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=8, thickness=12, line=dict(color="black", width=0.5),
                   label=all_labels),
        link=dict(source=sources, target=targets, value=values, color=colors_link)
    )])
    fig.update_layout(
        title="Sankey multinivel: metáforas convencionales → escenarios → regímenes",
        font_size=9, height=800
    )
    out_path = os.path.join(VIZ_DIR, "viz_sankey_multinivel.html")
    pio.write_html(fig, out_path, include_plotlyjs="cdn")
    print(f"✓ {out_path}")
    fig.show()
else:
    print("⚠ Sin datos suficientes para Sankey multinivel.")

## 7. Resumen

In [ ]:
# ============================================================
# 7. RESUMEN
# ============================================================
print("=" * 60)
print("N4 — VISUALIZACIONES COMPLETADAS")
print("=" * 60)
print(f"\nArchivos en {VIZ_DIR}:")
for f_name in sorted(os.listdir(VIZ_DIR)):
    size = os.path.getsize(os.path.join(VIZ_DIR, f_name)) / 1024
    icon = "📊" if f_name.endswith(".html") else "🖼"
    print(f"  {icon} {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: Ejecutar N5_narrativa_cultural.ipynb")